# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Oguzhandyr/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### 1. ML Task Formulation
* **Task Type:** Supervised Binary Classification & Learning-to-Rank (Scoring).
* **Core Objective:** Predict the probability of a URL entering a traffic/ranking decline within the observation window and produce a ranked queue sorted by expected impact.
* **Output Formulation:** $\hat{y} \in [0, 1]$ representing $P(\text{decay} \mid \mathbf{x})$, mapped to a continuous priority score $S = \hat{y} \times \log(\text{impressions\_90d} + 1)$.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### 2. Target Variable & Proxy Definition
* **Ground Truth / Proxy Target:** `is_declining_label` (Binary: `1` if `trend_direction == 'down'`, `0` otherwise).
* **Proxy Rationale:** Because exact real-time algorithmic penalties are unobservable, directional historical momentum (`trend_direction`) serves as a reliable proxy for decaying content value.
* **Leakage Safeguard:** Features derived directly from post-event trends (e.g., `trend_pct`) are strictly excluded from feature sets to prevent target leakage.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

### 3. Success Metrics (Offline & Decision Alignment)
* **Primary Metric:** **Precision@K (specifically Precision@20 and Precision@50)**.
  * *Why:* Content teams have finite review capacity (e.g., 20–50 URLs per cycle). Maximizing precision at the top of the ranked list ensures editorial resources are not wasted on false alarms.
* **Secondary Metric:** **ROC-AUC & Average Precision (PR-AUC)** to evaluate global ranking stability across class imbalance.
* **Business Evaluation:** Fraction of genuinely declining pages identified within the top-ranked audit batch vs. a baseline heuristic rule.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import os
import subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if "google.colab" in str(get_ipython()):
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    DATA_PATH = os.path.join(REPO_DIR, "data/raw/content_refresh_anonymized.csv")
else:
    DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

feature_cols = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
display_cols = feature_cols + ["is_declining_label"]

print(f"--- Dataset Schema & Unit of Analysis ---")
print(f"Total Rows (URLs): {len(df):,}")
print(f"Feature Vector Dimension: {len(feature_cols)}")
print(f"Target Distribution (Class 1 / Down): {df['is_declining_label'].mean():.2%}")
print("\nSample DataFrame Slice (First 5 Rows):")
df[display_cols].head()

--- Dataset Schema & Unit of Analysis ---
Total Rows (URLs): 30,000
Feature Vector Dimension: 6
Target Distribution (Class 1 / Down): 54.21%

Sample DataFrame Slice (First 5 Rows):


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,is_declining_label
0,187,20,3803,10.6,0.76,3221.0,1
1,445,25,15320,20.3,0.05,2481.0,1
2,141,20,12581,36.5,0.09,3515.0,1
3,463,22,11751,6.2,0.49,NaN,0
4,263,14,19140,44.0,0.13,2803.0,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### 5. Why ML Beats Fixed Heuristic Rules
1. **Non-linear Feature Interactions:** A simple rule like `days_since_last_update > 180 AND impressions > 500` fails when high-authority evergreen content stays stable despite age, or when recent pages rapidly lose ranking due to poor engagement. Decision trees and gradient boosting capture multi-way non-linear interactions automatically.
2. **Dynamic Priority Granularity:** Fixed thresholding produces a blunt binary group of tied pages without nuanced ordering. ML scoring outputs a calibrated probability distribution, enabling exact top-K prioritization.
3. **Adaptability Across Segments:** ML models adapt decision boundaries across varied position tiers and content types without needing manual hand-tuning of arbitrary cutoffs.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.